# Retrieval Pipeline Evaluation

Notebook for offline evaluation of both raw retrievers and the full pipeline variants built on top of them.


## Evaluation Plan

This notebook now measures two things separately:

1. baseline first-stage retrievers (`tfidf`, `bm25`, `embedding`)
2. pipeline variants for the configured final model

Pipeline variants include, when available:

- baseline retrieval
- category-filtered retrieval
- cross-encoder reranking
- cross-encoder reranking with category bonus
- category-filtered retrieval followed by reranking


In [8]:
import sys
from pathlib import Path

def detect_runtime_environment() -> str:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return 'colab'
    except Exception:
        if Path('/kaggle/input').exists():
            return 'kaggle'
        return 'local'

def add_project_root_to_syspath(project_name: str = 'retrieval_project') -> None:
    runtime_env = detect_runtime_environment()
    candidates = [Path.cwd(), *Path.cwd().parents]
    if runtime_env == 'colab':
        drive_root = Path('/content/drive/MyDrive')
        if not drive_root.exists():
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
        candidates = [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), *candidates]
    elif runtime_env == 'kaggle':
        candidates = [Path('/kaggle/working'), *candidates]

    seen = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base))
            return
        if (base / project_name / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base / project_name))
            return
        if runtime_env == 'colab' and base.exists():
            for match in base.rglob(project_name):
                if (match / 'src' / 'infra' / 'notebook.py').exists():
                    sys.path.insert(0, str(match))
                    return
    raise FileNotFoundError('Could not locate project root containing src/infra/notebook.py')

add_project_root_to_syspath()

import pandas as pd

from src.infra.notebook import setup_notebook
runtime_env, project_root = setup_notebook()
print(f'Detected runtime: {runtime_env}')
print(f'Project root    : {project_root}')


Detected runtime: colab
Project root    : /content/drive/MyDrive/retrieval_project


## Step 1: Load Data and Config

Load the normalized train/test views, the train ground truth, and the current pipeline configuration.


In [9]:
from dataclasses import replace

from src.config import DEFAULT_CONFIG
from src.evaluation import leaderboard_score, load_ground_truth
from src.pipeline import (
    bootstrap,
    build_cross_encoder_reranker,
    load_project_frames,
    predict_categories,
    prepare_retrievers,
    run_first_stage_retrieval,
)
from src.reranking import rerank_results_with_cross_encoder
from src.retrieval import run_retrieval, truncate_results

paths, config = bootstrap()
frames = load_project_frames(paths, DEFAULT_CONFIG)
ground_truth = load_ground_truth(paths.data_dir / 'qgts_train.json')
eval_top_k = max(config.retrieval_pipeline.evaluation_top_ks)

print(f'Documents          : {len(frames.docs):,}')
print(f'Train queries      : {len(frames.train_queries):,}')
print(f'Test queries       : {len(frames.test_queries):,}')
print(f'Ground truth       : {len(ground_truth):,}')
print(f'Final model        : {config.retrieval_pipeline.final_model}')
print(f'Evaluation models  : {config.retrieval_pipeline.evaluation_models}')
print(f'Evaluation top_ks  : {config.retrieval_pipeline.evaluation_top_ks}')
print(f'Category filter on : {config.retrieval_pipeline.enable_category_filter}')
print(f'Reranking on       : {config.retrieval_pipeline.enable_cross_encoder_rerank}')


Documents          : 216,041
Train queries      : 327
Test queries       : 141
Ground truth       : 327
Final model        : embedding
Evaluation models  : ('embedding',)
Evaluation top_ks  : (7500,)
Category filter on : True
Reranking on       : True


## Step 2: Prepare Shared Artifacts

Prepare first-stage retrieval artifacts, category predictions, and the cross-encoder if reranking is enabled.


In [10]:
prepared_retrievers = prepare_retrievers(frames, paths, config=config)
category_artifacts = predict_categories(frames, paths, ground_truth=ground_truth, config=config)
cross_encoder_reranker = build_cross_encoder_reranker(frames, paths, ground_truth, config=config)

print(f'Prepared retrievers: {sorted(prepared_retrievers.keys())}')
print(f'Category accuracy  : {category_artifacts.classifier_accuracy:.5f}')
print(f'Reranker ready     : {cross_encoder_reranker is not None}')


Loading cross-encoder from cache: cross-encoder_ms-marco-MiniLM-L6-v2_a25cb62561b05e78


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Prepared retrievers: ['embedding']
Category accuracy  : 0.92661
Reranker ready     : True


## Step 3: Baseline Retriever Comparison

This section keeps the old behavior: compare the raw first-stage retrievers directly.


In [11]:
baseline_results_by_model = {}
for model_name in sorted(set(config.retrieval_pipeline.evaluation_models) | {config.retrieval_pipeline.final_model}):
    print(f'Running baseline retrieval for: {model_name}')
    baseline_results_by_model[model_name] = run_retrieval(
        model_name=model_name,
        docs_frame=frames.docs,
        queries_frame=frames.train_queries,
        top_k=eval_top_k,
        cache_dir=paths.cache_dir,
        prepared_artifacts=prepared_retrievers[model_name],
        embedding_kind='queries_train',
        config=config,
    )


Running baseline retrieval for: embedding
Starting retrieval: model=embedding
  parameters: top_k=7,500, docs=216,041, queries=327, prepared_artifacts=yes, embedding_kind='queries_train'
  [Embedding] scoring 327 queries against 216,041 docs with capped_top_k=7,500, chunk_size=32, embedding_cache_key='queries_train'
  [Embedding] chunk 1/11: queries 1-32
  [Embedding] chunk 2/11: queries 33-64
  [Embedding] chunk 3/11: queries 65-96
  [Embedding] chunk 4/11: queries 97-128
  [Embedding] chunk 5/11: queries 129-160
  [Embedding] chunk 6/11: queries 161-192
  [Embedding] chunk 7/11: queries 193-224
  [Embedding] chunk 8/11: queries 225-256
  [Embedding] chunk 9/11: queries 257-288
  [Embedding] chunk 10/11: queries 289-320
  [Embedding] chunk 11/11: queries 321-327
Completed retrieval: model=embedding, results=327 queries, elapsed=3.5s


In [12]:
baseline_rows = []
for model_name, results in baseline_results_by_model.items():
    for top_k in config.retrieval_pipeline.evaluation_top_ks:
        metrics = leaderboard_score(
            truncate_results(results, top_k),
            ground_truth,
            k=top_k,
            accuracy_value=category_artifacts.classifier_accuracy,
        )
        baseline_rows.append({'Model': model_name, 'TopK': int(top_k), **metrics})

baseline_summary_df = pd.DataFrame(baseline_rows).sort_values(
    ['LeaderboardScore', 'MRR', 'Recall', 'TopK'],
    ascending=[False, False, False, False],
).reset_index(drop=True)
baseline_summary_df


,Model,TopK,Recall,Precision,MRR,Accuracy,LeaderboardScore
0,embedding,7500,0.947617,0.001099,0.474507,0.926606,0.587457


## Step 4: Pipeline Variant Comparison

This is the missing part: evaluate the configured final model under different pipeline options instead of only scoring the raw retriever.


In [13]:
query_category_map = category_artifacts.train_query_category_map or {}
doc_category_map = category_artifacts.doc_category_map or {}
has_category_filter = category_artifacts.classifier_artifacts is not None
has_reranker = cross_encoder_reranker is not None

base_config = replace(
    config,
    retrieval_pipeline=replace(
        config.retrieval_pipeline,
        enable_category_filter=False,
        enable_cross_encoder_rerank=False,
    ),
)

filtered_config = replace(
    config,
    retrieval_pipeline=replace(
        config.retrieval_pipeline,
        enable_category_filter=True,
        enable_cross_encoder_rerank=False,
    ),
)

variant_results = {}
variant_diagnostics = {}

baseline_pipeline_results, _ = run_first_stage_retrieval(
    frames=frames,
    paths=paths,
    prepared_retrievers=prepared_retrievers,
    category_artifacts=category_artifacts,
    split='train',
    top_k=eval_top_k,
    config=base_config,
)
variant_results['baseline'] = baseline_pipeline_results

if has_category_filter:
    category_filtered_results, _ = run_first_stage_retrieval(
        frames=frames,
        paths=paths,
        prepared_retrievers=prepared_retrievers,
        category_artifacts=category_artifacts,
        split='train',
        top_k=eval_top_k,
        config=filtered_config,
    )
    variant_results['category_filtered'] = category_filtered_results
else:
    category_filtered_results = None

if has_reranker:
    reranked_results, reranked_diagnostics = rerank_results_with_cross_encoder(
        results=baseline_pipeline_results,
        query_frame=frames.train_queries,
        docs_frame=frames.docs,
        cross_encoder=cross_encoder_reranker,
        query_category_map=query_category_map,
        doc_category_map=doc_category_map,
        infer_batch_size=config.cross_encoder.infer_batch_size,
        rerank_top_m=config.cross_encoder.rerank_top_m,
        category_bonus=0.0,
        return_diagnostics=True,
    )
    variant_results['reranked'] = reranked_results
    variant_diagnostics['reranked'] = reranked_diagnostics

    if query_category_map and doc_category_map:
        reranked_with_bonus_results, reranked_with_bonus_diagnostics = rerank_results_with_cross_encoder(
            results=baseline_pipeline_results,
            query_frame=frames.train_queries,
            docs_frame=frames.docs,
            cross_encoder=cross_encoder_reranker,
            query_category_map=query_category_map,
            doc_category_map=doc_category_map,
            infer_batch_size=config.cross_encoder.infer_batch_size,
            rerank_top_m=config.cross_encoder.rerank_top_m,
            category_bonus=config.cross_encoder.category_bonus,
            return_diagnostics=True,
        )
        variant_results['reranked_plus_category_bonus'] = reranked_with_bonus_results
        variant_diagnostics['reranked_plus_category_bonus'] = reranked_with_bonus_diagnostics

    if category_filtered_results is not None:
        filtered_reranked_results, filtered_reranked_diagnostics = rerank_results_with_cross_encoder(
            results=category_filtered_results,
            query_frame=frames.train_queries,
            docs_frame=frames.docs,
            cross_encoder=cross_encoder_reranker,
            query_category_map=query_category_map,
            doc_category_map=doc_category_map,
            infer_batch_size=config.cross_encoder.infer_batch_size,
            rerank_top_m=config.cross_encoder.rerank_top_m,
            category_bonus=config.cross_encoder.category_bonus if query_category_map and doc_category_map else 0.0,
            return_diagnostics=True,
        )
        variant_results['category_filtered_plus_rerank'] = filtered_reranked_results
        variant_diagnostics['category_filtered_plus_rerank'] = filtered_reranked_diagnostics

print('Evaluated variants:')
print(sorted(variant_results.keys()))


Starting retrieval: model=embedding
  parameters: top_k=7,500, docs=216,041, queries=327, prepared_artifacts=yes, embedding_kind='queries_train'
  [Embedding] scoring 327 queries against 216,041 docs with capped_top_k=7,500, chunk_size=32, embedding_cache_key='queries_train'
  [Embedding] chunk 1/11: queries 1-32
  [Embedding] chunk 2/11: queries 33-64
  [Embedding] chunk 3/11: queries 65-96
  [Embedding] chunk 4/11: queries 97-128
  [Embedding] chunk 5/11: queries 129-160
  [Embedding] chunk 6/11: queries 161-192
  [Embedding] chunk 7/11: queries 193-224
  [Embedding] chunk 8/11: queries 225-256
  [Embedding] chunk 9/11: queries 257-288
  [Embedding] chunk 10/11: queries 289-320
  [Embedding] chunk 11/11: queries 321-327
Completed retrieval: model=embedding, results=327 queries, elapsed=2.0s
Loading queries_gaming_filtered embeddings from cache: queries_gaming_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_0e045ca17ab5453a.npy
Loading queries_unix_filtered embeddings from cache: queries_u

In [14]:
variant_rows = []
for variant_name, results in variant_results.items():
    for top_k in config.retrieval_pipeline.evaluation_top_ks:
        metrics = leaderboard_score(
            truncate_results(results, top_k),
            ground_truth,
            k=top_k,
            accuracy_value=category_artifacts.classifier_accuracy,
        )
        variant_rows.append({'Variant': variant_name, 'TopK': int(top_k), **metrics})

variant_summary_df = pd.DataFrame(variant_rows)
baseline_scores = variant_summary_df[variant_summary_df['Variant'] == 'baseline'][['TopK', 'LeaderboardScore']].rename(columns={'LeaderboardScore': 'BaselineLeaderboardScore'})
variant_summary_df = variant_summary_df.merge(baseline_scores, on='TopK', how='left')
variant_summary_df['DeltaVsBaseline'] = variant_summary_df['LeaderboardScore'] - variant_summary_df['BaselineLeaderboardScore']
variant_summary_df = variant_summary_df.sort_values(
    ['LeaderboardScore', 'DeltaVsBaseline', 'MRR', 'Recall', 'TopK'],
    ascending=[False, False, False, False, False],
).reset_index(drop=True)
variant_summary_df


,Variant,TopK,Recall,Precision,MRR,Accuracy,LeaderboardScore,BaselineLeaderboardScore,DeltaVsBaseline
0,reranked_plus_category_bonus,7500,0.947617,0.001099,0.624435,0.926606,0.624939,0.587457,0.037482
1,reranked,7500,0.947617,0.001099,0.622633,0.926606,0.624489,0.587457,0.037032
2,category_filtered_plus_rerank,7500,0.880766,0.001073,0.584309,0.926606,0.598189,0.587457,0.010732
3,baseline,7500,0.947617,0.001099,0.474507,0.926606,0.587457,0.587457,0.000000
4,category_filtered,7500,0.880766,0.001073,0.450931,0.926606,0.564844,0.587457,-0.022613


In [15]:
variant_pivot = variant_summary_df.pivot(
    index='TopK',
    columns='Variant',
    values=['Recall', 'Precision', 'MRR', 'Accuracy', 'LeaderboardScore', 'DeltaVsBaseline'],
)
variant_pivot


Recall                                                            \
Variant  baseline category_filtered category_filtered_plus_rerank  reranked   
TopK                                                                          
7500     0.947617          0.880766                      0.880766  0.947617   

                                     Precision                    \
Variant reranked_plus_category_bonus  baseline category_filtered   
TopK                                                               
7500                        0.947617  0.001099          0.001073   

                                                                              \
Variant category_filtered_plus_rerank  reranked reranked_plus_category_bonus   
TopK                                                                           
7500                         0.001073  0.001099                     0.001099   

         ... LeaderboardScore                                                  \
Variant  ...         baseline category_filtered category_filtered_plus_rerank   
TopK     ...                                                                    
7500     ...         0.587457          0.564844                      0.598189   

                                               DeltaVsBaseline  \
Variant  reranked reranked_plus_category_bonus        baseline   
TopK                                                             
7500     0.624489                     0.624939             0.0   

                                                                   \
Variant category_filtered category_filtered_plus_rerank  reranked   
TopK                                                                
7500            -0.022613                      0.010732  0.037032   

                                      
Variant reranked_plus_category_bonus  
TopK                                  
7500                        0.037482  

[1 rows x 30 columns]

In [16]:
best_variant_row = variant_summary_df.iloc[0]
print('Best pipeline variant:')
print(best_variant_row.to_string())


Best pipeline variant:
Variant                     reranked_plus_category_bonus
TopK                                                7500
Recall                                          0.947617
Precision                                       0.001099
MRR                                             0.624435
Accuracy                                        0.926606
LeaderboardScore                                0.624939
BaselineLeaderboardScore                        0.587457
DeltaVsBaseline                                 0.037482
